In [4]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
data = pd.read_csv('exoTrain.csv')
X= data.drop('LABEL', axis=1)
y = data['LABEL']
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.long)
X = X.unsqueeze(1) # reshape for CNN
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

class CNN1D(nn.Module):
    def __init__(self):
        super(CNN1D, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5), # 1 input ki 16 output channels 
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2) #reduces computation
        )
        print(X.shape)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64), #lazy linear layer infers input size from data
            nn.ReLU(), 
            nn.Dropout(0.3)  
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

model = CNN1D()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):

    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)

    print(f"Epoch {epoch+1}, Train Loss: {loss.item()}, Val Loss: {val_loss.item()}")

    _, preds = torch.max(val_outputs, 1)
accuracy = (preds == y_val).float().mean()

print("Validation Accuracy:", accuracy.item())
print(torch.bincount(y_train))
print(torch.bincount(y_val))

torch.Size([5087, 1, 3197])
Epoch 1, Train Loss: 264.2240295410156, Val Loss: 76.83308410644531
Epoch 2, Train Loss: 6.54711389541626, Val Loss: 120.87285614013672
Epoch 3, Train Loss: 9.450952529907227, Val Loss: 152.92428588867188
Epoch 4, Train Loss: 12.4784517288208, Val Loss: 177.2401123046875
Epoch 5, Train Loss: 13.28746223449707, Val Loss: 195.93817138671875
Epoch 6, Train Loss: 14.953397750854492, Val Loss: 210.18089294433594
Epoch 7, Train Loss: 10.866724967956543, Val Loss: 221.1223907470703
Epoch 8, Train Loss: 12.917926788330078, Val Loss: 229.19752502441406
Epoch 9, Train Loss: 15.775556564331055, Val Loss: 234.64035034179688
Epoch 10, Train Loss: 13.765865325927734, Val Loss: 238.0189666748047
Epoch 11, Train Loss: 13.638516426086426, Val Loss: 239.61863708496094
Epoch 12, Train Loss: 14.724632263183594, Val Loss: 239.57977294921875
Epoch 13, Train Loss: 14.984424591064453, Val Loss: 238.12249755859375
Epoch 14, Train Loss: 11.681483268737793, Val Loss: 235.6688842773437

In [5]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')
X= data.drop('LABEL', axis=1)
y = data['LABEL']
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.long)
X = X.unsqueeze(1) # reshape for CNN
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Create data loaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class CNN1D(nn.Module):
    def __init__(self):
        super(CNN1D, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5), # 1 input ki 16 output channels 
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2) #reduces computation
        )
        print(X.shape)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64), #lazy linear layer infers input size from data
            nn.ReLU(), 
            nn.Dropout(0.3)  
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

model = CNN1D()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss}, Val Loss: {val_loss}") #updated training loop to use mini-batches instead of full batch training

    _, preds = torch.max(val_outputs, 1)
accuracy = (preds == y_val).float().mean()

print("Validation Accuracy:", accuracy.item())
print(torch.bincount(y_train))
print(torch.bincount(y_val))
#before full batch training resulting in fast memorization of training data and poor generalization to validation set. After implementing mini-batch training, the model learns more generalized patterns, leading to improved validation accuracy.

torch.Size([5087, 1, 3197])
Epoch 1, Train Loss: 857.9686512351036, Val Loss: 84.18476364202797
Epoch 2, Train Loss: 175.4408556232229, Val Loss: 4.5908025447279215
Epoch 3, Train Loss: 163.2794263958931, Val Loss: 3.6060910162050277
Epoch 4, Train Loss: 158.68816483020782, Val Loss: 3.5662441747263074
Epoch 5, Train Loss: 164.89730349183083, Val Loss: 3.9683378299087053
Epoch 6, Train Loss: 163.97140103578568, Val Loss: 3.7360534275794635
Epoch 7, Train Loss: 162.90310668945312, Val Loss: 3.892608130008739
Epoch 8, Train Loss: 160.8426183462143, Val Loss: 4.609068485155149
Epoch 9, Train Loss: 166.4979213476181, Val Loss: 198.5426757793066
Epoch 10, Train Loss: 160.54496650693682, Val Loss: 3.497802432742901
Epoch 11, Train Loss: 155.22933393716812, Val Loss: 4.046237817470683
Epoch 12, Train Loss: 163.46351528167725, Val Loss: 4.138726400648011
Epoch 13, Train Loss: 162.36543679237366, Val Loss: 4.0749987855379
Epoch 14, Train Loss: 166.16341900821135, Val Loss: 3.4614306506118737
Ep

In [10]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')
X= data.drop('LABEL', axis=1)
y = data['LABEL']
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.long)
X = X.unsqueeze(1) # reshape for CNN
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Create data loaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


class CNN1D(nn.Module):
    def __init__(self):
        super(CNN1D, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=5), # 1 input ki 16 output channels 
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(8, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2) #reduces computation
        )
        print(X.shape)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64), #lazy linear layer infers input size from data
            nn.ReLU(), 
            nn.Dropout(0.3)  
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

model = CNN1D()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss}, Val Loss: {val_loss}") #updated training loop to use mini-batches instead of full batch training

    _, preds = torch.max(val_outputs, 1)
accuracy = (preds == y_val).float().mean()

print("Validation Accuracy:", accuracy.item())
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_val.numpy(), preds.numpy()))
print(classification_report(y_val.numpy(), preds.numpy()))
print(torch.bincount(y_train))
print(torch.bincount(y_val))

#before full batch training resulting in fast memorization of training data and poor generalization to validation set. After implementing mini-batch training, the model learns more generalized patterns, leading to improved validation accuracy.

torch.Size([5087, 1, 3197])
Epoch 1, Train Loss: 808.4043822586536, Val Loss: 553.1083409076673
Epoch 2, Train Loss: 239.97432452440262, Val Loss: 18.559983417391777
Epoch 3, Train Loss: 176.76653042435646, Val Loss: 36.26103732745287
Epoch 4, Train Loss: 181.26757419109344, Val Loss: 8.06399621797027
Epoch 5, Train Loss: 168.1826889514923, Val Loss: 5.176674122834811
Epoch 6, Train Loss: 162.17939656972885, Val Loss: 95.88599291000583
Epoch 7, Train Loss: 206.67465955018997, Val Loss: 235.33897206788095
Epoch 8, Train Loss: 173.27109849452972, Val Loss: 9.95604984626192
Epoch 9, Train Loss: 175.9894408583641, Val Loss: 8.371123183381215
Epoch 10, Train Loss: 166.26530861854553, Val Loss: 8.71309342512048
Epoch 11, Train Loss: 169.78911620378494, Val Loss: 7.611356441072417
Epoch 12, Train Loss: 168.4277205169201, Val Loss: 7.958917841862018
Epoch 13, Train Loss: 168.2079439163208, Val Loss: 8.699852367559195
Epoch 14, Train Loss: 164.22548785805702, Val Loss: 11.101972372611378
Epoch 

c:\Users\sushanthboda\Downloads\New folder (2)\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sushanthboda\Downloads\New folder (2)\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sushanthboda\Downloads\New folder (2)\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [22]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')
X= data.drop('LABEL', axis=1)
y = data['LABEL'] - 1
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.long)
X = X.unsqueeze(1) # reshape for CNN
print(torch.unique(y))

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train distribution:", torch.bincount(y_train))
print("Val distribution:", torch.bincount(y_val))

# Create data loaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


class CNN1D(nn.Module):
    def __init__(self):
        super(CNN1D, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=5), # 1 input ki 16 output channels 
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(8, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2) #reduces computation
        )
        print(X.shape)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64), #lazy linear layer infers input size from data
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

class_counts = torch.bincount(y_train)
weights = 1.0 / class_counts.float()
weights = weights / weights.sum()

model = CNN1D()
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in val_loader:
            outputs = model(xb)

        all_preds.append(preds)
        all_labels.append(yb)
        
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

accuracy = (all_preds == all_labels).float().mean()

print("Validation Accuracy:", accuracy.item())
print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy()))
#before full batch training resulting in fast memorization of training data and poor generalization to validation set. After implementing mini-batch training, the model learns more generalized patterns, leading to improved validation accuracy.

tensor([0, 1])
Train distribution: tensor([4039,   30])
Val distribution: tensor([1011,    7])
torch.Size([5087, 1, 3197])
Validation Accuracy: 1.0
[[26]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        26

    accuracy                           1.00        26
   macro avg       1.00      1.00      1.00        26
weighted avg       1.00      1.00      1.00        26



c:\Users\sushanthboda\Downloads\New folder (2)\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train distribution:", torch.bincount(torch.tensor(y_train)))
print("Val distribution:", torch.bincount(torch.tensor(y_val)))

# convert to tensors AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(8, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = CNN1D()

class_counts = torch.bincount(y_train)
weights = 1.0 / class_counts.float()
weights = weights / weights.sum()

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy()))

Train distribution: tensor([4039,   30])
Val distribution: tensor([1011,    7])
Epoch 1, Train Loss: 42870.2429
Epoch 2, Train Loss: 1753.8991
Epoch 3, Train Loss: 910.2238
Epoch 4, Train Loss: 1030.1231
Epoch 5, Train Loss: 200.1715
Epoch 6, Train Loss: 152.5211
Epoch 7, Train Loss: 316.6968
Epoch 8, Train Loss: 112.4730
Epoch 9, Train Loss: 78.9921
Epoch 10, Train Loss: 29.5542
Epoch 11, Train Loss: 88.5014
Epoch 12, Train Loss: 23.3872
Epoch 13, Train Loss: 8.3846
Epoch 14, Train Loss: 4.5581
Epoch 15, Train Loss: 45.3643
Epoch 16, Train Loss: 6.5833
Epoch 17, Train Loss: 2.7594
Epoch 18, Train Loss: 12.8826
Epoch 19, Train Loss: 2.0595
Epoch 20, Train Loss: 2.2402
Total evaluated samples: 1018
Validation Accuracy: 0.9931237697601318
[[1010    1]
 [   6    1]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1011
           1       0.50      0.14      0.22         7

    accuracy                           0.99      1018
   macro

In [26]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train distribution:", torch.bincount(torch.tensor(y_train)))
print("Val distribution:", torch.bincount(torch.tensor(y_val)))

# convert to tensors AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(8, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = CNN1D()

class_counts = torch.bincount(y_train)
weights = torch.tensor([1.0, 50.0])
weights = weights / weights.sum()

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy())) #--only manual weights update

Train distribution: tensor([4039,   30])
Val distribution: tensor([1011,    7])
Epoch 1, Train Loss: 1344.3879
Epoch 2, Train Loss: 84.6418
Epoch 3, Train Loss: 73.9323
Epoch 4, Train Loss: 68.2863
Epoch 5, Train Loss: 63.8196
Epoch 6, Train Loss: 60.9842
Epoch 7, Train Loss: 59.9200
Epoch 8, Train Loss: 55.8649
Epoch 9, Train Loss: 55.3357
Epoch 10, Train Loss: 53.0859
Epoch 11, Train Loss: 53.9564
Epoch 12, Train Loss: 50.7947
Epoch 13, Train Loss: 50.8108
Epoch 14, Train Loss: 52.3521
Epoch 15, Train Loss: 51.2441
Epoch 16, Train Loss: 52.3790
Epoch 17, Train Loss: 842.7062
Epoch 18, Train Loss: 52.4099
Epoch 19, Train Loss: 49.1334
Epoch 20, Train Loss: 50.6900
Total evaluated samples: 1018
Validation Accuracy: 0.9931237697601318
[[1011    0]
 [   7    0]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1011
           1       0.00      0.00      0.00         7

    accuracy                           0.99      1018
   macro av

c:\Users\sushanthboda\Downloads\New folder (2)\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sushanthboda\Downloads\New folder (2)\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sushanthboda\Downloads\New folder (2)\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train distribution:", torch.bincount(torch.tensor(y_train)))
print("Val distribution:", torch.bincount(torch.tensor(y_val)))

# convert to tensors AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(8, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = CNN1D()

class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy()))

Train distribution: tensor([4039,   30])
Val distribution: tensor([1011,    7])
Epoch 1, Train Loss: 2162.3516
Epoch 2, Train Loss: 7.8586
Epoch 3, Train Loss: 3.4426
Epoch 4, Train Loss: 1.9032
Epoch 5, Train Loss: 13.3664
Epoch 6, Train Loss: 64.0314
Epoch 7, Train Loss: 17.8680
Epoch 8, Train Loss: 7.5134
Epoch 9, Train Loss: 46.7470
Epoch 10, Train Loss: 5.9479
Epoch 11, Train Loss: 7.0033
Epoch 12, Train Loss: 4.8275
Epoch 13, Train Loss: 6.4384
Epoch 14, Train Loss: 7.6769
Epoch 15, Train Loss: 4.0489
Epoch 16, Train Loss: 4.8615
Epoch 17, Train Loss: 3.5143
Epoch 18, Train Loss: 3.8021
Epoch 19, Train Loss: 4.1899
Epoch 20, Train Loss: 2.8825
Total evaluated samples: 1018
Validation Accuracy: 0.9646365642547607
[[981  30]
 [  6   1]]
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1011
           1       0.03      0.14      0.05         7

    accuracy                           0.96      1018
   macro avg       0.51      0.